In [ ]:
# IN09: Governance, Scaling, SLA/SLO, Resilience + Deployment Assessment

In [2]:
# Objectives

# By the end of this notebook you will be able to:
# - Implement data lineage tracking and structured audit trails for GenAI calls
# - Apply batching, streaming, and model-routing strategies to manage latency and cost
# - Define and measure SLA/SLO targets appropriate for a retail AI assistant
# - Build fallback model chains and graceful degradation for resilience
# - Consolidate checklist scores from IN07 and IN08 into a final deployment readiness assessment

# Deliverable: deployment_readiness_assessment.txt

In [3]:
import os, json, time, uuid, hashlib, statistics
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)
print('Client ready.')

Client ready.


In [4]:
# Main Business Problem
# Walmart may receive millions of AI-assisted customer interactions. Therefore, simply generating correct answers is not enough.

# The system must also answer questions such as:
# - Which model and prompt produced a particular answer?
# - Which documents were used as context?
# - Can simple questions use a faster and cheaper model?
# - Are latency and availability meeting agreed targets?
# - What happens when the primary model fails?
# - How do we stop one session from sending excessive requests?
# - Based on all controls, should deployment be approved or blocked?

# What the Notebook Builds?
# 1. Data governance and lineage
# 2. Scaling and latency management: Model routing, Streaming, Batching
# 3. SLA and SLO measurement: P50, P95 and P99 latency, Availability, Blocked-request rate, SLO breaches
# 4. Resilience and graceful degradation
# 5. Rate limiting
# 6. Final deployment-readiness assessment

In [5]:
# The notebook combines scores from:

# IN07: Reliability and observability
# IN08: Security
# IN09: Governance, scaling and resilience

# It generates the final deliverable: deployment_readiness_assessment.txt

## Data Governance -- Lineage, Versioning, and Audit Trails

Data governance for GenAI requires tracking the full provenance of every decision:

| What to track | Why |
|---|---|
| **Model version** | Reproducibility -- same input, same version = same output |
| **Prompt version** | Regression testing -- prompt changes affect all downstream answers |
| **Retrieved context** | Traceability -- which documents sourced the answer |
| **Temperature / params** | Audit compliance -- stochastic outputs must be explainable |
| **Latency per step** | SLO measurement and bottleneck identification |
| **User scope** | Compliance -- who asked what, when |

A lineage record persists beyond the request lifetime. It is the evidence trail
that satisfies an internal audit or a customer complaint about a factually wrong answer.

In [6]:
# {
#     "request_id": "REQ-1001",
#     "model_version": "gpt-4-turbo-2026-05",
#     "prompt_version": "walmart_prompt_v3",
#     "context_ids": ["policy_12", "product_7821"],
#     "temperature": 0.0,
#     "latency_ms": {
#         "retrieval": 320,
#         "llm": 1800,
#         "guardrails": 110
#     },
#     "user_scope": "user:9001"
# }

In [7]:
# DATA LINEAGE TRACKER

# The lineage tells us:
# - Which prompt version was used
# - Which model answered
# - Which context documents were provided
# - How many tokens were consumed
# - How long the call took
# - Whether it succeeded

LINEAGE_STORE = []  # In production: write to a persistent store (e.g. BigQuery, Delta Lake)

PROMPT_REGISTRY = {
    'v1.0': 'You are the Walmart Retail Assistant. Help customers with product and order queries.',
    'v1.1': (
        'You are the Walmart Retail Assistant. Help customers with product information, '
        'store policies, and order enquiries. Be concise and accurate. '
        'Do not discuss topics unrelated to Walmart products and services.'
    ),
}
ACTIVE_PROMPT_VERSION = 'v1.1'

def call_with_lineage(
    user_input: str,
    session_id: str,
    context_docs: list = None,
    model: str = 'gpt-4-turbo',
) -> dict:
    """Make an LLM call and record full data lineage."""
    request_id = str(uuid.uuid4())[:8]
    prompt_text = PROMPT_REGISTRY[ACTIVE_PROMPT_VERSION]
    start = time.time()

    messages = [
        {'role': 'system', 'content': prompt_text},
    ]
    if context_docs:
        ctx = '\n'.join(f'[{i+1}] {d}' for i, d in enumerate(context_docs))
        messages.append({'role': 'user', 'content': f'Context:\n{ctx}\n\nQuery: {user_input}'})
    else:
        messages.append({'role': 'user', 'content': user_input})

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.0,
            max_tokens=300,
            timeout=5,
        )
        answer = response.choices[0].message.content
        input_tokens  = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens
        success = True
    except Exception as e:
        answer = None
        input_tokens = output_tokens = 0
        success = False

    latency = round(time.time() - start, 3)

    lineage_record = {
        'request_id':      request_id,
        'session_id':      session_id,
        'timestamp':       time.time(),
        'model':           model,
        'model_provider':  'openai',
        'prompt_version':  ACTIVE_PROMPT_VERSION,
        'prompt_hash':     hashlib.sha256(prompt_text.encode()).hexdigest()[:12],
        'input_hash':      hashlib.sha256(user_input.encode()).hexdigest()[:12],
        'context_doc_ids': [hashlib.sha256(d.encode()).hexdigest()[:8] for d in (context_docs or [])],
        'temperature':     0.0,
        'input_tokens':    input_tokens,
        'output_tokens':   output_tokens,
        'latency_sec':     latency,
        'success':         success,
    }
    LINEAGE_STORE.append(lineage_record)

    return {'answer': answer, 'request_id': request_id, 'latency': latency, 'lineage': lineage_record}

# Test lineage tracking
context = [
    'Great Value Whole Milk 1 gallon is priced at $3.98 and located in Aisle 12.',
    'Walmart return policy: 90-day window for most items with receipt.',
]
result = call_with_lineage('What is the price of milk?', 'S-001', context_docs=context)
print('Response     :', result['answer'][:200])
print('Request ID   :', result['request_id'])
print('Latency      :', result['latency'], 's')
print('Lineage')
for k, v in result['lineage'].items():
    print(f'  {k:<20}: {v}')

Response     : The price of Great Value Whole Milk 1 gallon is $3.98.
Request ID   : 90871bdc
Latency      : 2.85 s
Lineage
  request_id          : 90871bdc
  session_id          : S-001
  timestamp           : 1784789798.579456
  model               : gpt-4-turbo
  model_provider      : openai
  prompt_version      : v1.1
  prompt_hash         : c52f0abc28b4
  input_hash          : 3e71ba51ca41
  context_doc_ids     : ['73b7a6cf', '308e23da']
  temperature         : 0.0
  input_tokens        : 100
  output_tokens       : 16
  latency_sec         : 2.85
  success             : True


## Scaling and Latency Management

Three complementary strategies for Walmart's 140M weekly customer interactions:

| Strategy | Description | When to use | Latency savings |
|---|---|---|---|
| **Batching** | Group N independent requests into one API call | Async, non-interactive workflows | Up to 50% throughput gain |
| **Streaming** | Return tokens as they arrive | Interactive chat, long responses | Perceived latency reduction |
| **Model routing** | Route simple queries to a small fast model | Mixed query complexity | 60-80% cost reduction on simple queries |

**Model routing decision:**
```
Query complexity LOW  --> gpt-4o-mini  (fast, cheap, sufficient)
Query complexity HIGH --> gpt-4-turbo  (accurate, necessary)
```
Complexity signals: multi-step, comparison, policy interpretation, order dispute.

In [8]:
# MODEL ROUTING: Route query to small or large model based on complexity

SIMPLE_MODEL  = 'gpt-4o-mini'
COMPLEX_MODEL = 'gpt-4-turbo'

COMPLEX_SIGNALS = [
    r'compare|versus|vs\.',
    r'why|explain|how does|what is the difference',
    r'dispute|complaint|escalat',
    r'policy|regulation|legal|refund',
    r'(multiple|several|all) (product|option|item)',
]

import re

def classify_complexity(query: str) -> str:
    """Return "simple" or "complex" based on query signals."""
    lower = query.lower()
    for pattern in COMPLEX_SIGNALS:
        if re.search(pattern, lower):
            return 'complex'
    return 'simple'

def routed_call(user_input: str, session_id: str) -> dict:
    complexity = classify_complexity(user_input)
    model = COMPLEX_MODEL if complexity == 'complex' else SIMPLE_MODEL
    result = call_with_lineage(user_input, session_id, model=model)
    result['complexity'] = complexity
    result['model_used'] = model
    return result

routing_tests = [
    'What aisle is milk in?',
    'Can you compare the nutritional profiles of Great Value Whole Milk vs 2% Milk?',
    'I want to dispute a charge on my order -- what is the refund policy?',
    'What is the store hours?',
]

print('Model Routing Results:')
routing_costs = {'simple': 0, 'complex': 0}
for q in routing_tests:
    r = routed_call(q, 'S-ROUTE')
    routing_costs[r['complexity']] += 1
    print(f'  [{r["complexity"].upper():7}] [{r["model_used"]}] {q[:60]}')
    print(f'            Latency: {r["latency"]}s')
print()
print(f'Simple queries: {routing_costs["simple"]} (used {SIMPLE_MODEL})')
print(f'Complex queries: {routing_costs["complex"]} (used {COMPLEX_MODEL})')

Model Routing Results:
  [SIMPLE ] [gpt-4o-mini] What aisle is milk in?
            Latency: 1.463s
  [COMPLEX] [gpt-4-turbo] Can you compare the nutritional profiles of Great Value Whol
            Latency: 16.183s
  [COMPLEX] [gpt-4-turbo] I want to dispute a charge on my order -- what is the refund
            Latency: 16.43s
  [SIMPLE ] [gpt-4o-mini] What is the store hours?
            Latency: 1.533s

Simple queries: 2 (used gpt-4o-mini)
Complex queries: 2 (used gpt-4-turbo)


In [10]:
# STREAMING: Show token-by-token output for interactive perceived latency

def stream_response(user_input: str) -> str:
    """Stream LLM response token by token. Returns full text."""
    print('Streaming: ', end='', flush=True)
    full_text = ''
    with client.chat.completions.stream(
        model=SIMPLE_MODEL,
        messages=[
            {'role': 'system', 'content': 'You are the Walmart Retail Assistant. Be concise.'},
            {'role': 'user',   'content': user_input},
        ],
        temperature=0.0,
        max_tokens=150,
    ) as stream:
        for text in stream.text_stream:
            print(text, end='', flush=True)
            full_text += text
    print()
    return full_text

print('Streaming response demo:')
def stream_response(user_input: str) -> str:
    """Stream LLM response token by token. Returns full text."""
    print('Streaming: ', end='', flush=True)
    full_text = ''
    stream = client.chat.completions.create(
        model=SIMPLE_MODEL,
        messages=[
            {'role': 'system', 'content': 'You are the Walmart Retail Assistant. Be concise.'},
            {'role': 'user',   'content': user_input},
        ],
        temperature=0.0,
        max_tokens=150,
        stream=True,
    )
    for chunk in stream:
        text = chunk.choices[0].delta.content or ''
        print(text, end='', flush=True)
        full_text += text
    print()
    return full_text

print('Streaming response demo:')
stream_response('What is the Walmart return policy in one sentence?')

Streaming response demo:
Streaming response demo:
Streaming: Walmart's return policy allows most items to be returned within 90 days of purchase with a receipt, while some electronics have a 30-day return window.


"Walmart's return policy allows most items to be returned within 90 days of purchase with a receipt, while some electronics have a 30-day return window."

## SLA/SLO Definition and Measurement

**SLA (Service Level Agreement):** contractual commitment to end users.
**SLO (Service Level Objective):** internal engineering target (stricter than SLA).

| Metric | SLO Target | SLA Threshold | Measurement |
|---|---|---|---|
| P50 latency | < 800ms | < 1200ms | Median of last 1000 calls |
| P95 latency | < 2000ms | < 3000ms | 95th percentile |
| P99 latency | < 4000ms | < 5000ms | 99th percentile |
| Availability | 99.9% | 99.5% | Successful calls / total calls |
| Hallucination rate | < 2% | < 5% | Sampled ground-check failures |
| Blocked request rate | < 1% | < 3% | Guardrail blocks / total calls |

**Error budget:** If SLO is 99.9% availability over 30 days = 43.8 minutes of downtime allowed.
Burn rate monitoring triggers alert when budget depletes faster than expected.

In [11]:
# SLO MEASUREMENT FRAMEWORK

SLO_TARGETS = {
    'p50_latency_ms': 800,
    'p95_latency_ms': 2000,
    'p99_latency_ms': 4000,
    'availability_pct': 99.9,
    'block_rate_pct': 1.0,
}

def measure_slo(call_records: list) -> dict:
    """
    call_records: list of dicts with keys latency_sec, success, blocked
    Returns SLO measurement report.
    """
    if not call_records:
        return {'error': 'No records to measure'}

    latencies_ms = [r['latency_sec'] * 1000 for r in call_records]
    latencies_ms.sort()
    n = len(latencies_ms)

    def percentile(data, p):
        idx = max(0, int(p / 100 * len(data)) - 1)
        return round(data[idx], 1)

    successes = sum(1 for r in call_records if r.get('success', True))
    blocks    = sum(1 for r in call_records if r.get('blocked', False))

    measurements = {
        'total_calls':     n,
        'p50_latency_ms':  percentile(latencies_ms, 50),
        'p95_latency_ms':  percentile(latencies_ms, 95),
        'p99_latency_ms':  percentile(latencies_ms, 99),
        'availability_pct': round(successes / n * 100, 3),
        'block_rate_pct':   round(blocks / n * 100, 3),
    }

    violations = {}
    for metric, target in SLO_TARGETS.items():
        measured = measurements.get(metric, 0)
        if 'latency' in metric:
            ok = measured <= target
        elif 'availability' in metric:
            ok = measured >= target
        else:
            ok = measured <= target
        violations[metric] = {'target': target, 'measured': measured, 'status': 'OK' if ok else 'BREACH'}

    measurements['slo_violations'] = violations
    return measurements

# Simulate 50 call records using lineage store + synthetic data
import random; random.seed(42)
SIMULATED_CALLS = [
    {'latency_sec': random.gauss(0.8, 0.3), 'success': True, 'blocked': False}
    for _ in range(45)
] + [
    {'latency_sec': random.gauss(3.5, 0.5), 'success': True, 'blocked': False}
    for _ in range(3)
] + [
    {'latency_sec': 0.1, 'success': False, 'blocked': False} for _ in range(1)
] + [
    {'latency_sec': 0.05, 'success': True, 'blocked': True} for _ in range(1)
]
# Ensure non-negative latencies
for c in SIMULATED_CALLS:
    c['latency_sec'] = max(0.05, c['latency_sec'])

report = measure_slo(SIMULATED_CALLS)
print('SLO Measurement Report (50 simulated calls):')
print(f'  Total calls      : {report["total_calls"]}')
print(f'  P50 latency      : {report["p50_latency_ms"]} ms')
print(f'  P95 latency      : {report["p95_latency_ms"]} ms')
print(f'  P99 latency      : {report["p99_latency_ms"]} ms')
print(f'  Availability     : {report["availability_pct"]} %')
print(f'  Block rate       : {report["block_rate_pct"]} %')
print()
print('SLO Compliance:')
for metric, result in report['slo_violations'].items():
    print(f'  [{result["status"]:6}] {metric:<22} target={result["target"]} measured={result["measured"]}')

SLO Measurement Report (50 simulated calls):
  Total calls      : 50
  P50 latency      : 812.5 ms
  P95 latency      : 1193.3 ms
  P99 latency      : 3739.2 ms
  Availability     : 98.0 %
  Block rate       : 2.0 %

SLO Compliance:
  [BREACH] p50_latency_ms         target=800 measured=812.5
  [OK    ] p95_latency_ms         target=2000 measured=1193.3
  [OK    ] p99_latency_ms         target=4000 measured=3739.2
  [BREACH] availability_pct       target=99.9 measured=98.0
  [BREACH] block_rate_pct         target=1.0 measured=2.0


## Resilience Patterns -- Fallback Chain and Graceful Degradation

Production LLM systems must survive primary model failures without exposing errors to users.

**Fallback chain for Walmart Retail Assistant:**

```
Attempt 1: gpt-4-turbo   (primary, highest quality)
    |-- timeout or error
    v
Attempt 2: gpt-4o-mini   (fallback 1, fast, cheaper)
    |-- timeout or error
    v
Attempt 3: Cached response lookup  (fallback 2, static knowledge base)
    |-- cache miss
    v
Attempt 4: Hard-coded graceful response  (always succeeds)
```

Each attempt has its own timeout. The chain is transparent to the caller.

In [12]:
# FALLBACK CHAIN IMPLEMENTATION

RESPONSE_CACHE = {
    'return policy': 'Walmart offers a 90-day return window for most items with a receipt.',
    'store hours':   'Most Walmart stores are open 6 AM to 11 PM. Check your local store at walmart.com.',
    'milk price':    'Great Value Whole Milk 1 gallon is approximately $3.98.',
}

GRACEFUL_DEFAULT = (
    'I am temporarily unable to process your request. '
    'Please visit walmart.com, use the Walmart app, or call 1-800-WALMART for assistance.'
)

def cached_lookup(query: str) -> str:
    lower = query.lower()
    for key, response in RESPONSE_CACHE.items():
        if key in lower:
            return response
    return None

def call_model(user_input: str, model: str, timeout: float) -> str:
    """Attempt one LLM call with given model and timeout. Raises on failure."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': 'You are the Walmart Retail Assistant. Be concise.'},
            {'role': 'user',   'content': user_input},
        ],
        temperature=0.0,
        max_tokens=150,
        timeout=timeout,
    )
    return response.choices[0].message.content

def fallback_chain(user_input: str) -> dict:
    trace = []

    # Attempt 1: Primary model
    try:
        answer = call_model(user_input, 'gpt-4-turbo', timeout=5.0)
        trace.append('gpt-4-turbo: SUCCESS')
        return {'answer': answer, 'source': 'gpt-4-turbo', 'trace': trace}
    except Exception as e:
        trace.append(f'gpt-4-turbo: FAILED ({type(e).__name__})')

    # Attempt 2: Fallback model
    try:
        answer = call_model(user_input, 'gpt-4o-mini', timeout=3.0)
        trace.append('gpt-4o-mini: SUCCESS')
        return {'answer': answer, 'source': 'gpt-4o-mini', 'trace': trace}
    except Exception as e:
        trace.append(f'gpt-4o-mini: FAILED ({type(e).__name__})')

    # Attempt 3: Cache
    cached = cached_lookup(user_input)
    if cached:
        trace.append('cache: HIT')
        return {'answer': cached, 'source': 'cache', 'trace': trace}
    trace.append('cache: MISS')

    # Attempt 4: Graceful default
    trace.append('graceful_default: USED')
    return {'answer': GRACEFUL_DEFAULT, 'source': 'graceful_default', 'trace': trace}

# Test the chain
chain_tests = [
    'What is the return policy for electronics?',
    'What are the store hours?',
]
print('Fallback Chain Tests:')
for q in chain_tests:
    r = fallback_chain(q)
    print(f'Q     : {q}')
    print(f'Source: {r["source"]}')
    print(f'Trace : {r["trace"]}')
    print(f'A     : {r["answer"][:150]}')
    print()

Fallback Chain Tests:
Q     : What is the return policy for electronics?
Source: gpt-4-turbo
Trace : ['gpt-4-turbo: SUCCESS']
A     : Walmart typically allows returns of electronics within 30 days of purchase. Items must be returned in the original manufacturer’s packaging and with a

Q     : What are the store hours?
Source: gpt-4-turbo
Trace : ['gpt-4-turbo: SUCCESS']
A     : Store hours can vary by location. To get the most accurate store hours, please visit the Walmart website or app and use the store locator to find the 



In [13]:
# RATE LIMITING (sliding window per session)
# Completes the one remaining checklist gap from IN07

from collections import defaultdict, deque

RATE_LIMIT_WINDOW_SEC = 60   # 1-minute window
RATE_LIMIT_MAX_CALLS  = 20   # max 20 calls per session per minute

SESSION_CALL_TIMES = defaultdict(deque)

def rate_limit_check(session_id: str) -> dict:
    now = time.time()
    window_start = now - RATE_LIMIT_WINDOW_SEC
    calls = SESSION_CALL_TIMES[session_id]

    # Evict calls outside the window
    while calls and calls[0] < window_start:
        calls.popleft()

    if len(calls) >= RATE_LIMIT_MAX_CALLS:
        return {
            'allowed': False,
            'calls_in_window': len(calls),
            'retry_after_sec': round(RATE_LIMIT_WINDOW_SEC - (now - calls[0]), 1),
        }

    calls.append(now)
    return {'allowed': True, 'calls_in_window': len(calls)}

# Simulate burst from one session
print('Rate Limiter Test (burst of 22 calls from session S-BURST):')
blocked_count = 0
for i in range(22):
    r = rate_limit_check('S-BURST')
    if not r['allowed']:
        blocked_count += 1
        if blocked_count == 1:
            print(f'  Call {i+1}: BLOCKED | calls_in_window={r["calls_in_window"]} | retry_after={r["retry_after_sec"]}s')
print(f'Total calls: 22 | Blocked: {blocked_count} | Allowed: {22 - blocked_count}')

Rate Limiter Test (burst of 22 calls from session S-BURST):
  Call 21: BLOCKED | calls_in_window=20 | retry_after=60.0s
Total calls: 22 | Blocked: 2 | Allowed: 20


## Deployment Readiness Assessment

This section consolidates all scores from IN07, IN08, and IN09 into a single
structured assessment document.

The assessment uses three domains:

| Domain | Source | Max Score |
|---|---|---|
| Reliability + Observability | IN07 production checklist | 12 |
| Security | IN08 security controls | 9 |
| Governance + Scaling + Resilience | IN09 (this notebook) | 9 |

**Total: 30 points**
- < 20: BLOCKED
- 20-25: CONDITIONAL -- fix mandatory items
- 26-30: APPROVED with monitoring

In [14]:
# Load scores from IN07 and IN08 if available, else use defaults

try:
    in07_data = json.loads(Path('in07_checklist_scores.json').read_text())
    in07_fixed_score = in07_data.get('fixed_score', 11)
    in07_failures    = in07_data.get('failures', ['rate_limiting'])
    in07_checklist   = in07_data.get('checklist', {})
    print(f'IN07 scores loaded: {in07_fixed_score}/12')
except FileNotFoundError:
    in07_fixed_score = 11
    in07_failures    = ['rate_limiting']
    in07_checklist   = {}
    print('IN07 scores: using defaults (file not found -- run IN07 first)')

try:
    in08_data = json.loads(Path('in08_security_scores.json').read_text())
    in08_score = sum(1 for v in in08_data.values() if v)
    in08_checks = in08_data
    print(f'IN08 scores loaded: {in08_score}/{len(in08_data)}')
except FileNotFoundError:
    in08_score = 9
    in08_checks = {}
    print('IN08 scores: using defaults (file not found -- run IN08 first)')

IN07 scores loaded: 11/12
IN08 scores loaded: 9/9


In [15]:
# IN09 GOVERNANCE + SCALING + RESILIENCE SELF-ASSESSMENT

IN09_CHECKS = {
    'data_lineage_tracking':   True,
    'prompt_versioning':       True,
    'structured_audit_trail':  True,
    'model_routing':           True,
    'streaming_enabled':       True,
    'slo_measurement':         True,
    'fallback_chain':          True,
    'rate_limiting':           True,   # completed in this notebook
    'graceful_degradation':    True,
}
in09_score = sum(1 for v in IN09_CHECKS.values() if v)
print(f'IN09 score: {in09_score}/{len(IN09_CHECKS)}')

IN09 score: 9/9


In [16]:
# GENERATE deployment_readiness_assessment.txt

from datetime import datetime

# Weighted scores for the 30-point rubric
# IN07: 12-point checklist -- normalise to 12
# IN08: 9-point security checks (capped at 9)
# IN09: 9-point governance checks (capped at 9)
in07_points = min(in07_fixed_score, 12)
in08_points = min(in08_score, 9)
in09_points = min(in09_score, 9)
total_points = in07_points + in08_points + in09_points
max_points   = 30

if total_points >= 26:
    verdict = 'APPROVED with continuous monitoring'
elif total_points >= 20:
    verdict = 'CONDITIONAL APPROVAL -- address remaining gaps before peak traffic'
else:
    verdict = 'BLOCKED -- critical gaps present, do not deploy'

lines = [
    'WALMART RETAIL ASSISTANT -- DEPLOYMENT READINESS ASSESSMENT',
    '=' * 60,
    f'Assessment Date  : {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    f'Assessor         : Walmart Global Tech Academy -- India Track',
    f'Application      : Walmart Retail Assistant (GenAI)',
    f'Programme Module : 3 -- Production-Ready AI Systems',
    '',
    'DOMAIN 1: RELIABILITY AND OBSERVABILITY (IN07)',
    '-' * 55,
    f'Score: {in07_points} / 12',
    '',
    'Checks:',
]

checklist_labels = {
    'input_validation':       'Input length and character validation',
    'injection_detection':    'Prompt injection detection',
    'no_architecture_leak':   'System prompt does not leak architecture',
    'low_temperature':        'Temperature <= 0.2 (deterministic)',
    'output_guardrail':       'Output guardrail (toxicity / PII filter)',
    'hallucination_check':    'Hallucination grounding check',
    'audit_log':              'Structured audit log per request',
    'graceful_fallback':      'Graceful fallback on API failure',
    'sla_timeout':            'SLA budget enforced (timeout per call)',
    'content_moderation':     'Content moderation on input and output',
    'pii_masking':            'PII masking before logging',
    'rate_limiting':          'Rate limiting per user / session',
}

for key, label in checklist_labels.items():
    status = in07_checklist.get(key, key not in in07_failures)
    lines.append(f'  [{"PASS" if status else "FAIL"}] {label}')

lines += [
    '',
    'DOMAIN 2: SECURITY CONTROLS (IN08)',
    '-' * 55,
    f'Score: {in08_points} / 9',
    '',
    'Checks:',
]

security_labels = {
    'injection_detection':     'Direct prompt injection detection',
    'jailbreak_resistance':    'Jailbreak resistance (patterns + never-list)',
    'pii_masking_input':       'PII masking on user input',
    'pii_masking_output':      'PII masking on LLM output',
    'pii_safe_logging':        'PII-safe audit logging (hashed)',
    'exfiltration_controls':   'Data exfiltration controls',
    'content_moderation':      'OpenAI Moderation API integrated',
    'anchored_system_prompt':  'Anchored system prompt (constraint repetition)',
    'indirect_injection_clean':'Indirect injection sanitiser (tool/RAG outputs)',
}

for key, label in security_labels.items():
    status = in08_checks.get(key, True)
    lines.append(f'  [{"PASS" if status else "FAIL"}] {label}')

lines += [
    '',
    'DOMAIN 3: GOVERNANCE, SCALING, SLA/SLO, RESILIENCE (IN09)',
    '-' * 55,
    f'Score: {in09_points} / 9',
    '',
    'Checks:',
]

governance_labels = {
    'data_lineage_tracking':   'Data lineage tracking per LLM call',
    'prompt_versioning':       'Prompt versioning and registry',
    'structured_audit_trail':  'Structured audit trail (request_id, hashes)',
    'model_routing':           'Model routing (complexity-based)',
    'streaming_enabled':       'Streaming enabled for interactive responses',
    'slo_measurement':         'SLO measurement framework (P50/P95/P99)',
    'fallback_chain':          'Fallback model chain implemented',
    'rate_limiting':           'Rate limiting (sliding window per session)',
    'graceful_degradation':    'Graceful degradation (cached + default responses)',
}

for key, label in governance_labels.items():
    status = IN09_CHECKS.get(key, False)
    lines.append(f'  [{"PASS" if status else "FAIL"}] {label}')

lines += [
    '',
    '=' * 60,
    'FINAL ASSESSMENT',
    '=' * 60,
    f'IN07 Reliability Score : {in07_points:>3} / 12',
    f'IN08 Security Score    : {in08_points:>3} / 9',
    f'IN09 Governance Score  : {in09_points:>3} / 9',
    f'TOTAL                  : {total_points:>3} / {max_points}',
    '',
    f'VERDICT: {verdict}',
    '',
    'MONITORING REQUIREMENTS (post-deployment):',
    '  1. P95 latency alert: > 2000ms triggers PagerDuty',
    '  2. Block rate alert:  > 1% triggers security review',
    '  3. Hallucination rate: sampled weekly via ground-check pipeline',
    '  4. PII alert log:     reviewed daily by data governance team',
    '  5. SLO burn rate:     30-day rolling dashboard (Grafana / Databricks)',
    '',
    'NEXT REVIEW DATE: 30 days post-deployment',
    '=' * 60,
]

report_text = '\n'.join(lines)
out_path = Path('deployment_readiness_assessment.txt')
out_path.write_text(report_text)
print(report_text)

WALMART RETAIL ASSISTANT -- DEPLOYMENT READINESS ASSESSMENT
Assessment Date  : 2026-07-23 12:48
Assessor         : Walmart Global Tech Academy -- India Track
Application      : Walmart Retail Assistant (GenAI)
Programme Module : 3 -- Production-Ready AI Systems

DOMAIN 1: RELIABILITY AND OBSERVABILITY (IN07)
-------------------------------------------------------
Score: 11 / 12

Checks:
  [PASS] Input length and character validation
  [PASS] Prompt injection detection
  [PASS] System prompt does not leak architecture
  [PASS] Temperature <= 0.2 (deterministic)
  [PASS] Output guardrail (toxicity / PII filter)
  [PASS] Hallucination grounding check
  [PASS] Structured audit log per request
  [PASS] Graceful fallback on API failure
  [PASS] SLA budget enforced (timeout per call)
  [PASS] Content moderation on input and output
  [PASS] PII masking before logging
  [FAIL] Rate limiting per user / session

DOMAIN 2: SECURITY CONTROLS (IN08)
--------------------------------------------------

In [17]:
print('Deliverable written: deployment_readiness_assessment.txt')
print(f'Total score: {total_points} / {max_points}')
print(f'Verdict    : {verdict}')

Deliverable written: deployment_readiness_assessment.txt
Total score: 29 / 30
Verdict    : APPROVED with continuous monitoring


## Summary

| Section | What was built |
|---|---|
| Data lineage | Full provenance record per LLM call (model, prompt version, hashes, latency) |
| Model routing | Complexity classifier routes to gpt-4o-mini or gpt-4-turbo |
| Streaming | Token-by-token streaming for interactive perceived latency |
| SLO framework | P50/P95/P99 latency + availability + block rate measurement |
| Fallback chain | 4-level degradation: primary model > fallback model > cache > static default |
| Rate limiting | Sliding 60-second window, 20 calls per session |
| Final assessment | `deployment_readiness_assessment.txt` across all three notebooks |

**Module 3 complete.** All three notebooks (IN07, IN08, IN09) together form
a production-readiness audit and implementation lab for the Walmart Retail Assistant.